# IsyaRasa — Fase 1: Training Model Klasifikasi Gloss

Notebook ini menjalankan seluruh Fase 1 dari PRD (bagian 15.2) secara end-to-end:
1. Unduh dataset **WL-BISINDO** dari Kaggle
2. Split **Signer-Independent** (SI)
3. Ekstraksi landmark tangan (MediaPipe)
4. Training model klasifikasi gloss (Conv1D + LSTM baseline)
5. Ekspor ke **TensorFlow.js**
6. Unduh hasil model untuk disalin ke `frontend/public/models/gloss-classifier/`

**Sebelum mulai:** Runtime → Change runtime type → pilih **GPU** (T4 cukup).

Logika di sini identik dengan script di `ml/dataset`, `ml/preprocessing`, `ml/training`, `ml/export` pada repo — supaya hasilnya kompatibel dengan `frontend/src/components/GlossClassifier.tsx` (urutan label, `SEQUENCE_LENGTH=30`, vector 126 dim per frame).

## 1. Install dependencies

*Catatan: versi tidak dipin secara ketat di sini — Colab sudah punya `tensorflow` bawaan, dan memaksa versi lama (`tensorflow==2.16.1` dkk seperti sebelumnya) sering bentrok dengan versi yang sudah terpasang di image Colab (error `ResolutionImpossible`). Kalau sel instalasi masih error setelah ini, jalankan **Runtime → Restart session**, lalu jalankan ulang dari sel ini.*

In [ ]:
!pip install -q -U mediapipe tensorflowjs kaggle tqdm

## 2. Kredensial Kaggle

Ambil `kaggle.json` dari **Kaggle → Account → Create New API Token**, lalu upload di sini.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()  # pilih kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
for fname in uploaded:
    if fname == 'kaggle.json':
        with open('/root/.kaggle/kaggle.json', 'wb') as f:
            f.write(uploaded[fname])
!chmod 600 /root/.kaggle/kaggle.json

## 3. Unduh dataset WL-BISINDO

In [ ]:
!mkdir -p /content/dataset/raw
!kaggle datasets download -d glennleonali/wl-bisindo -p /content/dataset/raw --unzip
!find /content/dataset/raw -name '*.mp4' | wc -l

## 4. Split Signer-Independent

Sama persis dengan `ml/dataset/organize_dataset.py` — signer 4 disisihkan untuk test (pengguna baru yang belum pernah dilihat model).

In [ ]:
import re, json, shutil
from pathlib import Path

FILENAME_RE = re.compile(r"signer(\d+)_label(\d+)_sample(\d+)\.mp4", re.IGNORECASE)
DEFAULT_SI_TEST_SIGNERS = [4]

def parse_filename(path):
    m = FILENAME_RE.match(path.name)
    if not m:
        return None
    return tuple(int(g) for g in m.groups())

def split_signer_independent(files, test_signers):
    train, test = [], []
    for f in files:
        parsed = parse_filename(f)
        if parsed is None:
            continue
        signer_id, _, _ = parsed
        (test if signer_id in test_signers else train).append(f)
    return train, test

def copy_split(files, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    for f in files:
        shutil.copy2(f, out_dir / f.name)

raw_dir = Path('/content/dataset/raw')
out_dir = Path('/content/dataset/split')
files_ = sorted(raw_dir.rglob('*.mp4'))
if not files_:
    raise SystemExit(f'Tidak ada file .mp4 ditemukan di {raw_dir} — cek hasil unduhan di sel sebelumnya')

train, test = split_signer_independent(files_, DEFAULT_SI_TEST_SIGNERS)
copy_split(train, out_dir / 'train')
copy_split(test, out_dir / 'test')
print(f'Split SI: {len(train)} train, {len(test)} test')

## 5. Ekstraksi landmark tangan (MediaPipe)

Sama persis dengan `ml/preprocessing/extract_landmarks.py` dan `landmarksToVector` di `GlossClassifier.tsx`: 2 tangan × 21 titik × (x,y,z) = 126 nilai per frame, tangan tak terdeteksi diisi nol.

In [ ]:
!wget -q -O /content/hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from tqdm import tqdm

NUM_HANDS = 2
NUM_POINTS = 21

def extract_video_landmarks(video_path, landmarker):
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_image)
        vector = np.zeros(NUM_HANDS * NUM_POINTS * 3, dtype=np.float32)
        for h, hand_landmarks in enumerate(result.hand_landmarks[:NUM_HANDS]):
            for i, point in enumerate(hand_landmarks[:NUM_POINTS]):
                offset = (h * NUM_POINTS + i) * 3
                vector[offset:offset + 3] = [point.x, point.y, point.z]
        frames.append(vector)
    cap.release()
    return np.stack(frames) if frames else np.zeros((0, NUM_HANDS * NUM_POINTS * 3), dtype=np.float32)

base_options = mp.tasks.BaseOptions(model_asset_path='/content/hand_landmarker.task')
options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_hands=NUM_HANDS,
)

landmarks_dir = Path('/content/landmarks')
with mp.tasks.vision.HandLandmarker.create_from_options(options) as landmarker:
    for split in ('train', 'test'):
        split_dir = out_dir / split
        if not split_dir.exists():
            continue
        out_split = landmarks_dir / split
        out_split.mkdir(parents=True, exist_ok=True)
        videos = sorted(split_dir.glob('*.mp4'))
        for video_path in tqdm(videos, desc=split):
            if not FILENAME_RE.match(video_path.name):
                continue
            sequence = extract_video_landmarks(video_path, landmarker)
            np.save(out_split / f'{video_path.stem}.npy', sequence)
print('Ekstraksi landmark selesai.')

## 6. Training

Sama persis dengan `ml/training/train.py` — baseline Conv1D+LSTM. **Urutan `LABELS` harus sama dengan `GLOSS_LABELS` di `GlossClassifier.tsx`** — jangan diubah tanpa menyamakan keduanya.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

LABELS = [
    "Air", "Belajar", "Cari", "Hari", "Ingat", "Lagi", "Maaf", "Makan",
    "Motor", "Saya", "Terima kasih", "Tuli", "Apa", "Siapa", "Kapan",
    "Di mana", "Mengapa", "Bagaimana", "Merah", "Kuning", "Hijau", "Hitam",
    "Dengar", "Berangkat", "Datang", "Teman", "Keluarga", "Rumah", "Pagi",
    "Siang", "Sore", "Malam",
]
SEQUENCE_LENGTH = 30
FEATURE_DIM = 126

def pad_or_trim(sequence, length):
    if len(sequence) >= length:
        return sequence[:length]
    pad = np.zeros((length - len(sequence), sequence.shape[1]), dtype=sequence.dtype)
    return np.concatenate([sequence, pad], axis=0)

def load_split(split_dir):
    X, y = [], []
    for npy_path in sorted(split_dir.glob('*.npy')):
        match = re.match(r"signer(\d+)_label(\d+)_sample(\d+)\.npy", npy_path.name, re.IGNORECASE)
        if not match:
            continue
        _, label_id, _ = (int(g) for g in match.groups())
        sequence = np.load(npy_path)
        X.append(pad_or_trim(sequence, SEQUENCE_LENGTH))
        y.append(label_id)
    return np.stack(X), np.array(y)

X_train, y_train = load_split(landmarks_dir / 'train')
X_test, y_test = load_split(landmarks_dir / 'test')
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

model = models.Sequential([
    layers.Input(shape=(SEQUENCE_LENGTH, FEATURE_DIM)),
    layers.Conv1D(64, 3, padding='same', activation='relu'),
    layers.Conv1D(64, 3, padding='same', activation='relu'),
    layers.LSTM(64, return_sequences=False),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(LABELS), activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

checkpoint_dir = Path('/content/checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    checkpoint_dir / 'best.keras', save_best_only=True, monitor='val_accuracy'
)

model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=16,
    callbacks=[checkpoint],
)

loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test accuracy (skema Signer-Independent): {accuracy:.4f}')

**Decision gate (PRD bagian 11 & 15.4):** kalau akurasi SI jauh di bawah target 80-90% (mis. < 60%), jangan lanjut ke ekspor dulu — pertimbangkan kurangi jumlah kosakata target (32 → 15-20 kata dengan akurasi tertinggi per label, lihat confusion matrix) sebelum commit ke model ini. Baseline Conv1D+LSTM di sini juga lebih sederhana dari Siformer/SPOTER (referensi PRD bagian 9) — kalau akurasi jauh di bawah ekspektasi, opsi lain adalah porting Siformer dari `AceKinnn/WL-BISINDO` ke sini.

## 7. Ekspor ke TensorFlow.js

In [ ]:
import tensorflowjs as tfjs
from tensorflow import keras

best_model = keras.models.load_model(checkpoint_dir / 'best.keras')
tfjs_out = Path('/content/gloss-classifier')
tfjs_out.mkdir(parents=True, exist_ok=True)
tfjs.converters.save_keras_model(best_model, str(tfjs_out))
print(f'Model TFJS disimpan di {tfjs_out}')
!ls -la /content/gloss-classifier

## 8. Unduh hasil

Setelah diunduh, ekstrak isi `gloss-classifier.zip` dan salin **isinya** (bukan foldernya sendiri) ke:
```
frontend/public/models/gloss-classifier/
```
sehingga ada `model.json` + file `.bin` langsung di dalam folder itu — sesuai `MODEL_URL = '/models/gloss-classifier/model.json'` di `GlossClassifier.tsx`.

In [ ]:
!zip -r -q /content/gloss-classifier.zip /content/gloss-classifier
from google.colab import files
files.download('/content/gloss-classifier.zip')